Análisis de la Información:
El dataset cuenta con datos históricos de consumo de cafe, contando con una columna para cada año desde 1990 a 2020, además de una columna para tipo de cafe, país y total de consumo doméstico, entonces la base de datos permite conocer información como el consumo de tipo de cafe específico para un año específico y un país específico, esto permitirá la predicción de consumo específico o general de cafe en un país determinado para un tiempo determinado.
Solución de la problemática usando técnicas de analítica:
Para solucionar el problema de negocio se implementaron tres métodos, una red neuronal que utiliza Softmax para poder predecir que tipos de cafe dominarán en determinados periódos y en determinados países, esta es una predicción categórica, el otro método es un modelo de predicción que busca predecir el consumo de cafe ya sea general o específico en un páis y un año determinado, finalmente, se implemento un LLM usando el transformer BERT para el procesamiento de lenguaje natural y RAG para el acceso de datos, esto permitiría realizar consultas en lenguaje natural.
Implementación y Evaluación de la solución:
Las soluciones son capaces de obtener datos históricos y realizar predicciones con base en los datos dados, el modelo de categorización tiene una precisión de alrededor de 0.95 es decir 95% según el entrenamiento y el modelo de predicción obtuvo un MAE de 2602315.8750 que aunque parece alto, esto se debe a que el MAE escala según los valores del dataset, por lo que en un dataset con valores altos (Como este que puede tener valores de hasta mil millones) se obtendrá un MAE alto, lo que significa es que las predicciones podrían tener un margen de error de 2602315.8750, lo cual no es mucho comparado a las magnitudes que maneja el dataset
La presentación de resultados se hace más adelante y se puede ver a través de Streamlit:
Resultado del modelo de clasificación
Probabilidades predichas: [0.08853287 0.02499491 0.02569171 0.8607805 ]
Tipo de café predicho (índice): 3
Tipo de café predicho: Robusta/Arabica
Tipo de café real: Robusta/Arabica
Resultados obtenidos por modelo de predicción
* Consumo general de café en 2021 para United States of America (estimado): 664,214,023.45
* Consumo de 'Arabica' en 2022 para Brazil: 782,969,241.96
* Consumo de 'Robusta' en 'Vietnam' en 2023: 68,503,034.37
* Consumo general de café en 'Germany' en 2024: 660,915,904.45
Resultados obtenidos del LLM
Consulta: Predice el consumo de Arabica en Brasil en 2025.
Resultado: Para predecir el consumo, necesito al menos un país y un año. ¿Podría proporcionar más detalles? (Falla porque se tienen en cuenta los países tal cual estan escritos en el dataset, al ser un dataset en inglés Brasil esta registrado como Brazil)

Consulta: ¿Cuánto café consumirá Alemania en 2024?
Resultado: Para predecir el consumo, necesito al menos un país y un año. ¿Podría proporcionar más detalles? (Falla porque Alemania no esta en el dataset y solo se tienen en cuenta los países del dataset)

Consulta: ¿Cuál es el tipo de café en Colombia en 2010?
Resultado: Para Colombia en el año 2010, el tipo de café predilecto es: **Arabica** con una probabilidad del 98.01%.

Consulta: Consumo de Robusta en Vietnam en el año 2018.
Resultado: El consumo histórico de café Robusta en 2018 fue de **59,505,240.00** unidades.
(Promedio anual si aplica: 3,967,016.00)

Consulta: Consumo de café en Estados Unidos en 2015.
Resultado: El consumo histórico de café 2015 fue de **2,840,952,000.00** unidades.
(Promedio anual si aplica: 51,653,672.73)

Consulta: Predice el consumo para 2030.
Resultado: Para predecir el consumo, necesito al menos un país y un año. ¿Podría proporcionar más detalles? (Falla porque falta el país)

Consulta: Consumo de café en Burundi en 1990.
Resultado: El consumo histórico de café Burundi en 1990 fue de **120,000.00** unidades.

In [1]:
import pandas as pd

# Cargar el dataset
try:
    df_coffee = pd.read_parquet('coffee_db.parquet')
    print("Dataset 'coffee_db.parquet' cargado exitosamente.\n")

    # Mostrar las primeras filas del DataFrame
    print("Primeras 5 filas del dataset:\n")
    print(df_coffee.head())
    print("\n")

    # Mostrar información general del DataFrame (tipos de datos, valores no nulos)
    print("Información general del dataset:\n")
    df_coffee.info()
    print("\n")

    # Mostrar estadísticas descriptivas para columnas numéricas
    print("Estadísticas descriptivas del dataset:\n")
    print(df_coffee.describe())

except FileNotFoundError:
    print("Error: El archivo 'coffee_db.parquet' no se encuentra. Asegúrate de que está en la ubicación correcta.")
except Exception as e:
    print(f"Ocurrió un error al cargar o procesar el archivo: {e}")

Dataset 'coffee_db.parquet' cargado exitosamente.

Primeras 5 filas del dataset:

                            Country      Coffee type    1990/91    1991/92  \
0                            Angola  Robusta/Arabica    1200000    1800000   
1  Bolivia (Plurinational State of)          Arabica    1500000    1620000   
2                            Brazil  Arabica/Robusta  492000000  510000000   
3                           Burundi  Arabica/Robusta     120000      96000   
4                           Ecuador  Arabica/Robusta   21000000   21000000   

     1992/93    1993/94    1994/95    1995/96    1996/97    1997/98  ...  \
0    2100000    1200000    1500000     600000    1200000    2400000  ...   
1    1650000    1710000    1770000    1830000    1890000    1950000  ...   
2  534000000  546000000  558000000  606000000  660000000  690000000  ...   
3     102000     114600     120000     120000     120000     120000  ...   
4   21000000   21000000   21000000   21000000   18000000   18000000  

En esta celda simplemente se carga el dataset y se revisa la información que contiene

In [3]:
# Identificar las columnas que representan los años de consumo
year_cols = [col for col in df_coffee.columns if '/' in col]

# Las columnas que no son años serán las de identificación
id_vars = ['Country', 'Coffee type', 'Total_domestic_consumption']

# Fundir el DataFrame de formato ancho a largo
df_long = pd.melt(df_coffee, id_vars=id_vars, value_vars=year_cols,
                  var_name='Year_Period', value_name='Consumption')

# Convertir la columna 'Consumption' a numérica. Coerción de errores a NaN.
df_long['Consumption'] = pd.to_numeric(df_long['Consumption'], errors='coerce')

# Extraer el año de inicio del periodo (ej. '1990/91' -> 1990)
df_long['Year'] = df_long['Year_Period'].apply(lambda x: int(x.split('/')[0]))

# Eliminar la columna original 'Year_Period' si ya no es necesaria
df_long = df_long.drop(columns=['Year_Period'])

# Reordenar las columnas para mayor claridad (opcional)
df_long = df_long[['Country', 'Coffee type', 'Year', 'Consumption', 'Total_domestic_consumption']]

print("\n--- Dataset Limpiado (primeras 5 filas) ---\n")
print(df_long.head())

print("\n--- Información general del Dataset Limpiado ---\n")
df_long.info()

print("\n--- Estadísticas descriptivas del Dataset Limpiado ---\n")
print(df_long.describe())

# Verificar valores nulos después de la limpieza
print("\n--- Valores nulos por columna después de la limpieza ---\n")
print(df_long.isnull().sum())


--- Dataset Limpiado (primeras 5 filas) ---

                            Country      Coffee type  Year  Consumption  \
0                            Angola  Robusta/Arabica  1990      1200000   
1  Bolivia (Plurinational State of)          Arabica  1990      1500000   
2                            Brazil  Arabica/Robusta  1990    492000000   
3                           Burundi  Arabica/Robusta  1990       120000   
4                           Ecuador  Arabica/Robusta  1990     21000000   

   Total_domestic_consumption  
0                    46500000  
1                    75180000  
2                 27824700000  
3                     3412020  
4                   381540000  

--- Información general del Dataset Limpiado ---

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1650 entries, 0 to 1649
Data columns (total 5 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Country                     1650 non-

Aquí se prepara el dataset para utilizarse en nuestros modelos, para esto se identifican las columnas de años (Que contienen los valores de consumo para cada año) y se definen las demás columnas como de identificación (Country, Coffee type y Total_domestic_consumption), luego se usa melt para fundir el dataset, lo que quiere decir que se crea una columna de año y una de consumo para poder distribuir el consumo por año de forma que sea más fácil de procesar, básicamente se expande el dataset para que clasifique por pais, año, tipo, consumo y total, luego se convierten las variables para facilitar su uso asegurandose de que sean números, luego se obtienen algunas estadísticas

In [4]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import numpy as np

# --- Preparación para el Modelo de Clasificación (Softmax) ---

# Definir las características (X) y la variable objetivo (y) para la clasificación
X_cls = df_long[['Country', 'Year']]
y_cls = df_long['Coffee type']

# Eliminar filas con valores nulos en X_cls o y_cls que puedan afectar el entrenamiento
# (Esto es especialmente importante si 'Consumption' tenía NaN, aunque para clasif. no es la target)
data_cls = pd.concat([X_cls, y_cls], axis=1).dropna()
X_cls = data_cls[['Country', 'Year']]
y_cls = data_cls['Coffee type']

# 1. Codificación de la variable objetivo 'Coffee type' (Label Encoding para luego One-Hot)
label_encoder = LabelEncoder()
y_cls_encoded = label_encoder.fit_transform(y_cls)

# Convertir a formato One-Hot para la salida de Softmax
onehot_encoder = OneHotEncoder(sparse_output=False)
y_cls_onehot = onehot_encoder.fit_transform(y_cls_encoded.reshape(-1, 1))

# Guardar las clases originales para referencia futura
coffee_type_classes = label_encoder.classes_

print("\n--- Preparación para Clasificación: Tipos de Café Codificados ---\n")
print(f"Clases de Coffee type: {coffee_type_classes}")
print(f"Primeros 5 valores de y_cls_encoded: {y_cls_encoded[:5]}")
print(f"Primeros 5 valores de y_cls_onehot:\n{y_cls_onehot[:5]}")

# 2. Codificación de características categóricas (Country)
# Usaremos ColumnTransformer para aplicar OneHotEncoder a 'Country'
# y dejar 'Year' como está.

# Identificar columnas categóricas para X_cls
categorical_features_cls = ['Country']

preprocessor_cls = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_cls)
    ],
    remainder='passthrough'
)

# Aplicar el preprocesamiento a X_cls
X_cls_processed = preprocessor_cls.fit_transform(X_cls)

print("\n--- Preparación para Clasificación: Características X procesadas ---\n")
print(f"Forma de X_cls original: {X_cls.shape}")
print(f"Forma de X_cls_processed (con One-Hot Encoding): {X_cls_processed.shape}")
print(f"Primeras filas de X_cls_processed (ejemplo, puede ser disperso):\n{X_cls_processed[:5].toarray() if hasattr(X_cls_processed, 'toarray') else X_cls_processed[:5]}")

# 3. División de datos para el modelo de clasificación
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls_processed, y_cls_onehot, test_size=0.2, random_state=42, stratify=y_cls_encoded
)

print("\n--- División de Datos para Clasificación ---\n")
print(f"X_train_cls shape: {X_train_cls.shape}")
print(f"X_test_cls shape: {X_test_cls.shape}")
print(f"y_train_cls shape: {y_train_cls.shape}")
print(f"y_test_cls shape: {y_test_cls.shape}")



--- Preparación para Clasificación: Tipos de Café Codificados ---

Clases de Coffee type: ['Arabica' 'Arabica/Robusta' 'Robusta' 'Robusta/Arabica']
Primeros 5 valores de y_cls_encoded: [3 0 1 1 1]
Primeros 5 valores de y_cls_onehot:
[[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]]

--- Preparación para Clasificación: Características X procesadas ---

Forma de X_cls original: (1650, 2)
Forma de X_cls_processed (con One-Hot Encoding): (1650, 56)
Primeras filas de X_cls_processed (ejemplo, puede ser disperso):
[[1.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00
  0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00
  0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00
  0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00
  0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00
  0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00
  0.00e+00 0.00

Ahora si se realiza el preprocesado para el entrenamiento de los modelos, primero se definen las variables independientes y la dependiente, en este caso, será Coffee Type, pues queremos predecir el tipo de cafe dominante para un determinado país en un determinado año, luego se eliminan los valores nulos, luego se codifican las variables y a través de OneHot se los convierte de datos categóricos a numeros binarios para Softmax, se guardan las clases originales, se aplica el preprocesamiento y luego se dividen los datos en prueba y entrenamiento para el modelo

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# --- Construcción del Modelo de Clasificación (Red Neuronal con Softmax) ---

# Obtener el número de características de entrada y el número de clases de salida
input_shape_cls = X_train_cls.shape[1]
output_classes_cls = y_train_cls.shape[1]

# Definir el modelo Sequential
model_cls = Sequential([
    Dense(64, activation='relu', input_shape=(input_shape_cls,)), # Primera capa oculta
    Dense(128, activation='relu'),                                  # Segunda capa oculta
    Dense(output_classes_cls, activation='softmax')                # Capa de salida con Softmax
])

# Compilar el modelo
model_cls.compile(
    optimizer='adam',
    loss='categorical_crossentropy', # Para clasificación multiclase con etiquetas one-hot
    metrics=['accuracy']
)

# Mostrar un resumen del modelo
print("\n--- Resumen del Modelo de Clasificación ---\n")
model_cls.summary()

# --- Entrenamiento del Modelo de Clasificación ---

print("\n--- Entrenamiento del Modelo de Clasificación ---\n")
history_cls = model_cls.fit(
    X_train_cls,
    y_train_cls,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_cls, y_test_cls),
    verbose=1
)

# --- Evaluación del Modelo de Clasificación ---

print("\n--- Evaluación del Modelo de Clasificación en el conjunto de prueba ---\n")
loss_cls, accuracy_cls = model_cls.evaluate(X_test_cls, y_test_cls, verbose=0)
print(f"Pérdida (Loss) en el conjunto de prueba: {loss_cls:.4f}")
print(f"Precisión (Accuracy) en el conjunto de prueba: {accuracy_cls:.4f}")

# --- Realizar predicciones (ejemplo) ---

# Tomar una muestra del conjunto de prueba para predecir
sample_index = np.random.randint(0, X_test_cls.shape[0])
sample_X_cls = X_test_cls[sample_index:sample_index+1]
sample_y_true_onehot = y_test_cls[sample_index:sample_index+1]

# Predecir las probabilidades
predictions_probabilities = model_cls.predict(sample_X_cls)
predicted_class_index = np.argmax(predictions_probabilities, axis=1)[0]
true_class_index = np.argmax(sample_y_true_onehot, axis=1)[0]

predicted_coffee_type = coffee_type_classes[predicted_class_index]
true_coffee_type = coffee_type_classes[true_class_index]

print("\n--- Ejemplo de Predicción ---\n")
print(f"Clases de Café: {coffee_type_classes}")
print(f"Probabilidades predichas: {predictions_probabilities[0]}")
print(f"Tipo de café predicho (índice): {predicted_class_index}")
print(f"Tipo de café predicho: {predicted_coffee_type}")
print(f"Tipo de café real: {true_coffee_type}")


--- Resumen del Modelo de Clasificación ---



Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │         3,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,484 (48.77 KB)

 Trainable params: 12,484 (48.77 KB)

 Non-trainable params: 0 (0.00 B)


--- Entrenamiento del Modelo de Clasificación ---

Epoch 1/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.2629 - loss: 38.6284 - val_accuracy: 0.4000 - val_loss: 11.0126
Epoch 2/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.2924 - loss: 3.9026 - val_accuracy: 0.1455 - val_loss: 3.8319
Epoch 3/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.3136 - loss: 4.4468 - val_accuracy: 0.1818 - val_loss: 10.5858
Epoch 4/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.2924 - loss: 6.8067 - val_accuracy: 0.1818 - val_loss: 1.5547
Epoch 5/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.2750 - loss: 3.8531 - val_accuracy: 0.1455 - val_loss: 3.5703
Epoch 6/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3023 - loss: 5.5011 - val_accuracy: 0.2727 - val_loss: 10.1709
Epoch 7/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3159 - loss: 7.4287 - val_accuracy: 0.4000 - val_loss: 9.0950
Epoch 8/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - ac

Se obtienen las características de entrada y las de salida, luego se crea el modelo secuencial utilizando dos capas ocultas con Relu que utilizan 64 y 128 unidades para la dimenssionalidad del espacio de salida y una capa de salida que usa Softmax, se compila el modelo usando un optimizador y una pérdida predefinidas, categorical_crossentropy porque es muy util para clasificación multiclase, y luego se entrena usando 50 epocas y los conjuntos de entrenamiento, finalmente se realizan predicciones de prueba que hace parte de la presentación de resultados, realizando una predicción de forma correcta

In [13]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import numpy as np

# --- Preparación para el Modelo de Regresión ---

# Definir las características (X) y la variable objetivo (y) para la regresión
X_reg = df_long[['Country', 'Coffee type', 'Year']]
y_reg = df_long['Consumption']

# Eliminar filas con valores nulos en la variable objetivo o características
# Esto es crucial para la regresión de consumo
data_reg = pd.concat([X_reg, y_reg], axis=1).dropna()
X_reg = data_reg[['Country', 'Coffee type', 'Year']]
y_reg = data_reg['Consumption']

# 1. Codificación de características categóricas (Country y Coffee type) y escalado de 'Year'
# Usaremos ColumnTransformer para aplicar OneHotEncoder a 'Country' y 'Coffee type'
# y StandardScaler a 'Year'.

categorical_features_reg = ['Country', 'Coffee type']
numerical_features_reg = ['Year']

preprocessor_reg = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_reg),
        ('num', StandardScaler(), numerical_features_reg) # Escalar la columna 'Year'
    ],
    remainder='drop' # Descartar cualquier otra columna no especificada
)

# Aplicar el preprocesamiento a X_reg
X_reg_processed = preprocessor_reg.fit_transform(X_reg)

# 2. Escalar la variable objetivo (y_reg)
scaler_y_reg = StandardScaler()
y_reg_scaled = scaler_y_reg.fit_transform(y_reg.values.reshape(-1, 1))

print("\n--- Preparación para Regresión: Características X procesadas ---\n")
print(f"Forma de X_reg original: {X_reg.shape}")
print(f"Forma de X_reg_processed (con One-Hot Encoding y Escalado): {X_reg_processed.shape}")
print(f"Primeras filas de X_reg_processed (ejemplo, puede ser disperso):\n{X_reg_processed[:5]}")

print("\n--- Preparación para Regresión: Variable objetivo y procesada ---\n")
print(f"Forma de y_reg original: {y_reg.shape}")
print(f"Forma de y_reg_scaled: {y_reg_scaled.shape}")
print(f"Primeros 5 valores de y_reg_scaled:\n{y_reg_scaled[:5].flatten()}")

# 3. División de datos para el modelo de regresión (usando los datos escalados)
X_train_reg, X_test_reg, y_train_reg_scaled, y_test_reg_scaled = train_test_split(
    X_reg_processed, y_reg_scaled, test_size=0.2, random_state=42
)

print("\n--- División de Datos para Regresión ---\n")
print(f"X_train_reg shape: {X_train_reg.shape}")
print(f"X_test_reg shape: {X_test_reg.shape}")
print(f"y_train_reg_scaled shape: {y_train_reg_scaled.shape}")
print(f"y_test_reg_scaled shape: {y_test_reg_scaled.shape}")


--- Preparación para Regresión: Características X procesadas ---

Forma de X_reg original: (1650, 3)
Forma de X_reg_processed (con One-Hot Encoding y Escalado): (1650, 60)
Primeras filas de X_reg_processed (ejemplo, puede ser disperso):
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 15 stored elements and shape (5, 60)>
  Coords	Values
  (0, 0)	1.0
  (0, 58)	1.0
  (0, 59)	-1.6752467319482305
  (1, 1)	1.0
  (1, 55)	1.0
  (1, 59)	-1.6752467319482305
  (2, 2)	1.0
  (2, 56)	1.0
  (2, 59)	-1.6752467319482305
  (3, 3)	1.0
  (3, 56)	1.0
  (3, 59)	-1.6752467319482305
  (4, 13)	1.0
  (4, 56)	1.0
  (4, 59)	-1.6752467319482305

--- Preparación para Regresión: Variable objetivo y procesada ---

Forma de y_reg original: (1650,)
Forma de y_reg_scaled: (1650, 1)
Primeros 5 valores de y_reg_scaled:
[-0.2696358  -0.26738151  3.41839659 -0.27775128 -0.1208521 ]

--- División de Datos para Regresión ---

X_train_reg shape: (1320, 60)
X_test_reg shape: (330, 60)
y_train_reg_scaled shape: (

Se realiza la preparación para el entrenamiento del modelo de clasificación, de igual forma se definen las variables independientes y la variable dependiente, en este caso es consumption porque queremos predecir el consumo de cafe según país, categoría y año, se realiza el procedimiento estándar, eliminando nulos, luego se estandarizan los datos, usando OneHot encoder para variables categóricas y estandarizando los años, se aplica el preprocesamiento y el escalado y se dividen los datos en entrenamiento y pruebas

In [ ]:
# --- Construcción del Modelo de Regresión (Red Neuronal con Activación Lineal) ---

# Obtener el número de características de entrada
input_shape_reg = X_train_reg.shape[1]

# Definir el modelo Sequential
model_reg = Sequential([
    Dense(128, activation='relu', input_shape=(input_shape_reg,)), # Primera capa oculta
    Dense(64, activation='relu'),                                  # Segunda capa oculta
    Dense(1, activation='linear')                                  # Capa de salida con activación lineal para regresión
])

# Compilar el modelo
model_reg.compile(
    optimizer='adam',
    loss='mean_squared_error', # Para regresión, el error cuadrático medio es común
    metrics=['mean_absolute_error', 'mean_squared_error']
)

# Mostrar un resumen del modelo
print("\n--- Resumen del Modelo de Regresión ---\n")
model_reg.summary()

# --- Entrenamiento del Modelo de Regresión (usando datos escalados) ---

print("\n--- Entrenamiento del Modelo de Regresión ---\n")
history_reg = model_reg.fit(
    X_train_reg,
    y_train_reg_scaled, # Usar variable objetivo escalada
    epochs=50,
    batch_size=32,
    validation_data=(X_test_reg, y_test_reg_scaled), # Usar variable objetivo escalada
    verbose=1
)

# --- Evaluación del Modelo de Regresión (con predicciones desescaladas) ---

print("\n--- Evaluación del Modelo de Regresión en el conjunto de prueba ---\n")
# Predecir valores escalados
y_pred_reg_scaled = model_reg.predict(X_test_reg, verbose=0)

# Desescalar las predicciones y los valores reales para calcular las métricas en la escala original
y_pred_reg = scaler_y_reg.inverse_transform(y_pred_reg_scaled)
y_test_reg_original = scaler_y_reg.inverse_transform(y_test_reg_scaled)

# Calcular métricas de error en la escala original
mae_reg = np.mean(np.abs(y_pred_reg - y_test_reg_original))
mse_reg = np.mean((y_pred_reg - y_test_reg_original)**2)

print(f"Error Absoluto Medio (MAE) en el conjunto de prueba (desescalado): {mae_reg:.4f}")
print(f"Error Cuadrático Medio (MSE) en el conjunto de prueba (desescalado): {mse_reg:.4f}")

# --- Realizar predicciones para los escenarios solicitados ---

# Función auxiliar para preprocesar nuevas entradas de predicción
def preprocess_new_data_for_reg(country, coffee_type, year):
    new_data = pd.DataFrame({
        'Country': [country],
        'Coffee type': [coffee_type],
        'Year': [year]
    })
    # Usar el preprocessor_reg ya ajustado para transformar la entrada
    processed_data = preprocessor_reg.transform(new_data)
    return processed_data

print("\n--- Ejemplos de Predicción de Consumo ---\n")

example_year = 2021
example_country_for_general = 'United States of America'

all_coffee_types = df_long['Coffee type'].unique()

total_consumption_2021_usa = 0
for c_type in all_coffee_types:
    new_input = preprocess_new_data_for_reg(example_country_for_general, c_type, example_year)
    predicted_consumption_scaled = model_reg.predict(new_input, verbose=0)[0][0]
    # Desescalar la predicción
    predicted_consumption = scaler_y_reg.inverse_transform([[predicted_consumption_scaled]])[0][0]
    total_consumption_2021_usa += predicted_consumption

print(f"* Consumo general de café en {example_year} para {example_country_for_general} (estimado): {total_consumption_2021_usa:,.2f}")

example_coffee_type = 'Arabica'
example_year_2 = 2022
example_country_2 = 'Brazil'

new_input_2 = preprocess_new_data_for_reg(example_country_2, example_coffee_type, example_year_2)
predicted_consumption_2_scaled = model_reg.predict(new_input_2, verbose=0)[0][0]
predicted_consumption_2 = scaler_y_reg.inverse_transform([[predicted_consumption_2_scaled]])[0][0]
print(f"* Consumo de '{example_coffee_type}' en {example_year_2} para {example_country_2}: {predicted_consumption_2:,.2f}")

example_coffee_type_3 = 'Robusta'
example_country_3 = 'Vietnam'
example_year_3 = 2023

new_input_3 = preprocess_new_data_for_reg(example_country_3, example_coffee_type_3, example_year_3)
predicted_consumption_3_scaled = model_reg.predict(new_input_3, verbose=0)[0][0]
predicted_consumption_3 = scaler_y_reg.inverse_transform([[predicted_consumption_3_scaled]])[0][0]
print(f"* Consumo de '{example_coffee_type_3}' en '{example_country_3}' en {example_year_3}: {predicted_consumption_3:,.2f}")

example_country_4 = 'Germany'
example_year_4 = 2024

total_consumption_2024_germany = 0
for c_type in all_coffee_types:
    new_input = preprocess_new_data_for_reg(example_country_4, c_type, example_year_4)
    predicted_consumption_scaled = model_reg.predict(new_input, verbose=0)[0][0]
    predicted_consumption = scaler_y_reg.inverse_transform([[predicted_consumption_scaled]])[0][0]
    total_consumption_2024_germany += predicted_consumption

print(f"* Consumo general de café en '{example_country_4}' en {example_year_4}: {total_consumption_2024_germany:,.2f}")


--- Resumen del Modelo de Regresión ---



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 128)            │         7,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,129 (63.00 KB)

 Trainable params: 16,129 (63.00 KB)

 Non-trainable params: 0 (0.00 B)


--- Entrenamiento del Modelo de Regresión ---

Epoch 1/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.9373 - mean_absolute_error: 0.2876 - mean_squared_error: 0.9373 - val_loss: 0.4723 - val_mean_absolute_error: 0.2020 - val_mean_squared_error: 0.4723
Epoch 2/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.5886 - mean_absolute_error: 0.2000 - mean_squared_error: 0.5886 - val_loss: 0.1619 - val_mean_absolute_error: 0.1224 - val_mean_squared_error: 0.1619
Epoch 3/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.1688 - mean_absolute_error: 0.1118 - mean_squared_error: 0.1688 - val_loss: 0.0249 - val_mean_absolute_error: 0.0966 - val_mean_squared_error: 0.0249
Epoch 4/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0386 - mean_absolute_error: 0.0919 - mean_squared_error: 0.0386 - val_loss: 0.0116 - val_mean_absolute_error: 0.0681 - val_mean_squared_error: 0.0116
Epoch 5/50
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0126 - mean_absolute_error: 0.0601 - mean_squar

Ahora si se crea el modelo de predicción, se obtiene el número de características de entrada, se define el modelo secuencial con 2 capas ocultas con 128 y 64 unidades y función de activación Relu, y luego una capa de salida lineal debido a que se trata de una predicción que se puede realizar con regresión, se realiza la compilación, ahora la función de pérdida es MSE, común para regresión, y se definen MAE y MSE como métricas, se entrena el modelo usando las variables de entrenamiento, las de pruebas y 50 épocas, usando la versión escalada de la variable dependiente, esto permite mejorar el cálculo de la precisión del modelo a través del MSE y MAE, se realiza el test realizando predicción en los valores escalados y luego se deescala las predicciones para que se den en el valor que corresponde a la escala original, se crea una función para preprocesar las solicitudes y luego se realizan predicciones de prueba que permiten presentar los resultados del modelo

In [12]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# --- Configuración inicial de BERT para el LLM ---

# Vamos a cargar un tokenizador y un modelo BERT pre-entrenado.
# Un modelo como 'bert-base-uncased' es un buen punto de partida para entender el lenguaje.
# Para tareas de Question Answering, podemos usar modelos específicos.
# Por ejemplo, 'deepset/bert-base-cased-squad2' es un modelo fine-tuneado para QA.

print("\n--- Cargando Tokenizador y Modelo BERT ---\n")

tokenizer = AutoTokenizer.from_pretrained("deepset/bert-base-cased-squad2")
model_bert = AutoModelForQuestionAnswering.from_pretrained("deepset/bert-base-cased-squad2")

print("Tokenizador BERT cargado exitosamente.")
print("Modelo BERT para Question Answering cargado exitosamente.")

# --- Ejemplo de cómo BERT podría procesar una pregunta ---
# Este es un paso conceptual. La integración completa requeriría más lógica
# para mapear la salida de BERT a la entrada de nuestros modelos ML.

context = "El consumo de café Arábica en Brasil en 2022 fue de 1500000 unidades y en 2023 se predice 1600000 unidades."
question = "¿Cuánto fue el consumo de café Arábica en Brasil en 2022?"

# Codificar la pregunta y el contexto
inputs = tokenizer(question, context, return_tensors="pt")

# Obtener las predicciones del modelo (índices de inicio y fin de la respuesta)
with torch.no_grad():
    outputs = model_bert(**inputs)

answer_start_index = torch.argmax(outputs.start_logits)
answer_end_index = torch.argmax(outputs.end_logits) + 1

answer_tokens = inputs.input_ids[0, answer_start_index:answer_end_index]
answer = tokenizer.decode(answer_tokens)

print("\n--- Ejemplo de Procesamiento de Consulta con BERT (Question Answering) ---\n")
print(f"Contexto: {context}")
print(f"Pregunta: {question}")
print(f"Respuesta de BERT: {answer}")

print("\nEste ejemplo demuestra cómo BERT puede extraer una respuesta de un texto dado. Para nuestro caso, la idea sería:")
print("1. BERT analizaría la pregunta del usuario (ej. \"Consumo de Robusta en Vietnam en 2023\").")
print("2. Extraería las entidades clave (Robusta, Vietnam, 2023) y la intención (predicción de consumo).")
print("3. Estas entidades se usarían para invocar a nuestra función `preprocess_new_data_for_reg` y `model_reg.predict`.")
print("4. Finalmente, el LLM formularía una respuesta legible usando el valor predicho.")



--- Cargando Tokenizador y Modelo BERT ---



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizador BERT cargado exitosamente.
Modelo BERT para Question Answering cargado exitosamente.

--- Ejemplo de Procesamiento de Consulta con BERT (Question Answering) ---

Contexto: El consumo de café Arábica en Brasil en 2022 fue de 1500000 unidades y en 2023 se predice 1600000 unidades.
Pregunta: ¿Cuánto fue el consumo de café Arábica en Brasil en 2022?
Respuesta de BERT: [CLS] ¿ Cuánto fue el consumo de café Arábica en Brasil en 2022? [SEP] El consumo de café Arábica en Brasil en 2022 fue de 1500000 unidades y en 2023 se predice 1600000 unidades.

Este ejemplo demuestra cómo BERT puede extraer una respuesta de un texto dado. Para nuestro caso, la idea sería:
1. BERT analizaría la pregunta del usuario (ej. "Consumo de Robusta en Vietnam en 2023").
2. Extraería las entidades clave (Robusta, Vietnam, 2023) y la intención (predicción de consumo).
3. Estas entidades se usarían para invocar a nuestra función `preprocess_new_data_for_reg` y `model_reg.predict`.
4. Finalmente, el LLM form

Se carga el modelo BERT y su tokenizador luego se realiza una consulta al LLM de prueba, esto se realiza para verificar que el modelo opera

In [22]:
import re

def extract_entities_and_intent(query, countries, coffee_types):
    intent = None
    country = None
    coffee_type = None
    year = None

    # Convertir la consulta a minúsculas para facilitar la búsqueda
    query_lower = query.lower()

    # 1. Extracción de Año
    # Buscar patrones de 4 dígitos que puedan ser años (e.g., 2023, 1995)
    year_matches = re.findall(r'\b\d{4}\b', query_lower)
    if year_matches:
        # Priorizar años más recientes si hay varios, o el primero encontrado
        year = int(max(year_matches)) # Suponemos que el usuario pregunta por el año más relevante

    # 2. Extracción de País
    # Buscar nombres de países en la consulta (coincidencia parcial o exacta)
    # Asegurarse de que 'countries' sea una lista de cadenas
    for c in countries:
        if c.lower() in query_lower:
            country = c
            break

    # 3. Extracción de Tipo de Café
    # Buscar tipos de café en la consulta (coincidencia parcial o exacta)
    # Asegurarse de que 'coffee_types' sea una lista de cadenas
    for ct in coffee_types:
        if ct.lower() in query_lower:
            coffee_type = ct
            break

    # 4. Inferencia de Intención
    # Palabras clave para predicción
    prediction_keywords = ['predecir', 'estimar', 'pronosticar', 'futuro', 'predice', 'estima', 'pronostica']
    if any(keyword in query_lower for keyword in prediction_keywords):
        intent = 'predict_consumption'
    # Palabras clave para clasificación de tipo de café
    elif any(keyword in query_lower for keyword in ['tipo de café', 'clasificar tipo']) and coffee_type is None:
        intent = 'classify_coffee_type'
    # Si no es predicción ni clasificación, y tiene entidades, asumimos consulta histórica
    elif (country or coffee_type or year) and ('consumo' in query_lower or 'cantidad' in query_lower or 'fue' in query_lower):
        intent = 'historical_consumption'
    # Por defecto, si no hay palabras clave claras, pero hay entidades para predicción
    elif (country or coffee_type or year) and not intent:
        intent = 'predict_consumption' # Por defecto, si hay datos para predecir, asumimos esa intención
    else:
        intent = 'general_info'

    return {
        'intent': intent,
        'country': country,
        'coffee_type': coffee_type,
        'year': year
    }

# Ejemplo de uso de la función
# Para usar esta función, necesitamos los arrays 'all_coffee_types' y 'df_long['Country'].unique()'
# que ya han sido definidos en celdas anteriores.

unique_countries = df_long['Country'].unique()
unique_coffee_types = all_coffee_types

query1 = "¿Cuál será el consumo de café Arabica en Brasil en 2025?"
extracted1 = extract_entities_and_intent(query1, unique_countries, unique_coffee_types)
print(f"Consulta: {query1}\nExtracción: {extracted1}\n")

query2 = "Consumo de café en Vietnam en 2018?"
extracted2 = extract_entities_and_intent(query2, unique_countries, unique_coffee_types)
print(f"Consulta: {query2}\nExtracción: {extracted2}\n")

query3 = "Quiero saber el tipo de café de Burundi en 2000."
extracted3 = extract_entities_and_intent(query3, unique_countries, unique_coffee_types)
print(f"Consulta: {query3}\nExtracción: {extracted3}\n")

query4 = "Predice el consumo total en Alemania para el 2024."
extracted4 = extract_entities_and_intent(query4, unique_countries, unique_coffee_types)
print(f"Consulta: {query4}\nExtracción: {extracted4}\n")

query5 = "Datos de consumo para el año 2010 en Colombia."
extracted5 = extract_entities_and_intent(query5, unique_countries, unique_coffee_types)
print(f"Consulta: {query5}\nExtracción: {extracted5}\n")

Consulta: ¿Cuál será el consumo de café Arabica en Brasil en 2025?
Extracción: {'intent': 'historical_consumption', 'country': None, 'coffee_type': 'Arabica', 'year': 2025}

Consulta: Consumo de café en Vietnam en 2018?
Extracción: {'intent': 'historical_consumption', 'country': None, 'coffee_type': None, 'year': 2018}

Consulta: Quiero saber el tipo de café de Burundi en 2000.
Extracción: {'intent': 'classify_coffee_type', 'country': 'Burundi', 'coffee_type': None, 'year': 2000}

Consulta: Predice el consumo total en Alemania para el 2024.
Extracción: {'intent': 'predict_consumption', 'country': None, 'coffee_type': None, 'year': 2024}

Consulta: Datos de consumo para el año 2010 en Colombia.
Extracción: {'intent': 'historical_consumption', 'country': 'Colombia', 'coffee_type': None, 'year': 2010}



Se importa re para el procesamiento de expresiones regulares, usamos una función que se encargará de extraer entidades e intención, la query se pasa a minúsculas para facilitar el procesamiento, luego se configura para que el sistema busque expresiones que se asemejen a la expresión regular que representa números de cuatro digitos, luego se busca el nombre de algún país, comparando las palabras con un conjunto de países, lo mismo se hace con tipos de cafe, luego se infiere la intención a través de palabras clave, estableciendo que la query tiene uno de 5 tipos de intención que el sistema usará para responder, ya sea prediciendo, dando información histórica, o clasificando, los tipos de cafe y de país se obtienen buscando los valores únicos de las columnas en el dataset, luego se realizan queries de prueba para poder verificar que se identifica la intención

In [23]:
import pandas as pd
import numpy as np

def query_ml_models(extracted_info):
    intent = extracted_info['intent']
    country = extracted_info['country']
    coffee_type = extracted_info['coffee_type']
    year = extracted_info['year']

    response = "Lo siento, no pude procesar su consulta. Asegúrese de proporcionar un país y un año para las predicciones de consumo, o un país y un año para clasificar el tipo de café, o los datos necesarios para una consulta histórica."

    # --- Manejo de la intención de clasificación (Classify Coffee Type) ---
    if intent == 'classify_coffee_type':
        if country and year:
            # Preprocesar la entrada para el modelo de clasificación
            new_data_cls = pd.DataFrame({'Country': [country], 'Year': [year]})
            processed_data_cls = preprocessor_cls.transform(new_data_cls)

            # Realizar la predicción
            predictions_probabilities = model_cls.predict(processed_data_cls)
            predicted_class_index = np.argmax(predictions_probabilities, axis=1)[0]
            predicted_coffee_type = coffee_type_classes[predicted_class_index]

            response = f"Para {country} en el año {year}, el tipo de café predilecto es: **{predicted_coffee_type}** con una probabilidad del {predictions_probabilities[0][predicted_class_index]*100:.2f}%."
        else:
            response = "Para clasificar un tipo de café, necesito un país y un año. ¿Podría proporcionar más detalles?"

    # --- Manejo de la intención de predicción de consumo (Predict Consumption) ---
    elif intent == 'predict_consumption':
        if country and year:
            if coffee_type:
                # Predicción de consumo para un tipo de café específico en un país y año dados
                new_input = preprocess_new_data_for_reg(country, coffee_type, year)
                predicted_consumption_scaled = model_reg.predict(new_input, verbose=0)[0][0]
                predicted_consumption = scaler_y_reg.inverse_transform([[predicted_consumption_scaled]])[0][0]
                response = f"Se predice que el consumo de **{coffee_type}** en **{country}** para el año **{year}** será de aproximadamente **{predicted_consumption:,.2f}** unidades."
            else:
                # Predicción de consumo total para un país y año dados (suma de todos los tipos)
                total_consumption = 0
                for c_type in all_coffee_types:
                    new_input = preprocess_new_data_for_reg(country, c_type, year)
                    predicted_consumption_scaled = model_reg.predict(new_input, verbose=0)[0][0]
                    predicted_consumption = scaler_y_reg.inverse_transform([[predicted_consumption_scaled]])[0][0]
                    total_consumption += predicted_consumption
                response = f"Se predice que el consumo general de café en **{country}** para el año **{year}** será de aproximadamente **{total_consumption:,.2f}** unidades."
        else:
            response = "Para predecir el consumo, necesito al menos un país y un año. ¿Podría proporcionar más detalles?"

    # --- Manejo de la intención de consumo histórico (Historical Consumption) ---
    elif intent == 'historical_consumption':
        # Filtrar el DataFrame df_long para obtener datos históricos
        filtered_df = df_long.copy()
        if country: filtered_df = filtered_df[filtered_df['Country'].str.lower() == country.lower()]
        if coffee_type: filtered_df = filtered_df[filtered_df['Coffee type'].str.lower() == coffee_type.lower()]
        if year: filtered_df = filtered_df[filtered_df['Year'] == year]

        if not filtered_df.empty:
            total_historical_consumption = filtered_df['Consumption'].sum()
            details = []
            if country: details.append(country)
            if coffee_type: details.append(coffee_type)
            if year: details.append(str(year))
            details_str = " en ".join(details)

            response = f"El consumo histórico de café {details_str} fue de **{total_historical_consumption:,.2f}** unidades.\n"
            if len(filtered_df) > 1:
                response += "(Promedio anual si aplica: " + f"{filtered_df['Consumption'].mean():,.2f})"
        else:
            response = "No se encontraron datos históricos para los criterios proporcionados. Por favor, intente con otra consulta."

    return response

print("--- Ejemplos de Consultas al Controlador ML ---")

# Ejemplo 1: Predicción de consumo específico
query_ex1 = "Predice el consumo de Arabica en Brasil en 2025."
extracted_ex1 = extract_entities_and_intent(query_ex1, unique_countries, unique_coffee_types)
print(f"Consulta: {query_ex1}\nResultado: {query_ml_models(extracted_ex1)}\n")

# Ejemplo 2: Predicción de consumo total para un país y año
query_ex2 = "¿Cuánto café consumirá Alemania en 2024?"
extracted_ex2 = extract_entities_and_intent(query_ex2, unique_countries, unique_coffee_types)
print(f"Consulta: {query_ex2}\nResultado: {query_ml_models(extracted_ex2)}\n")

# Ejemplo 3: Clasificación de tipo de café
query_ex3 = "¿Cuál es el tipo de café en Colombia en 2010?"
extracted_ex3 = extract_entities_and_intent(query_ex3, unique_countries, unique_coffee_types)
print(f"Consulta: {query_ex3}\nResultado: {query_ml_models(extracted_ex3)}\n")

# Ejemplo 4: Consulta histórica
query_ex4 = "Consumo de Robusta en Vietnam en el año 2018."
extracted_ex4 = extract_entities_and_intent(query_ex4, unique_countries, unique_coffee_types)
print(f"Consulta: {query_ex4}\nResultado: {query_ml_models(extracted_ex4)}\n")

# Ejemplo 5: Consulta histórica general (sin tipo de café específico)
query_ex5 = "Consumo de café en Estados Unidos en 2015."
extracted_ex5 = extract_entities_and_intent(query_ex5, unique_countries, unique_coffee_types)
print(f"Consulta: {query_ex5}\nResultado: {query_ml_models(extracted_ex5)}\n")

# Ejemplo 6: Consulta sin suficientes datos para el modelo
query_ex6 = "Predice el consumo para 2030."
extracted_ex6 = extract_entities_and_intent(query_ex6, unique_countries, unique_coffee_types)
print(f"Consulta: {query_ex6}\nResultado: {query_ml_models(extracted_ex6)}\n")

# Ejemplo 7: Consulta histórica con múltiples resultados (ejemplo, si hubiera varios tipos en el mismo país/año)
query_ex7 = "Consumo de café en Burundi en 1990."
extracted_ex7 = extract_entities_and_intent(query_ex7, unique_countries, unique_coffee_types)
print(f"Consulta: {query_ex7}\nResultado: {query_ml_models(extracted_ex7)}\n")

--- Ejemplos de Consultas al Controlador ML ---
Consulta: Predice el consumo de Arabica en Brasil en 2025.
Resultado: Para predecir el consumo, necesito al menos un país y un año. ¿Podría proporcionar más detalles?

Consulta: ¿Cuánto café consumirá Alemania en 2024?
Resultado: Para predecir el consumo, necesito al menos un país y un año. ¿Podría proporcionar más detalles?

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
Consulta: ¿Cuál es el tipo de café en Colombia en 2010?
Resultado: Para Colombia en el año 2010, el tipo de café predilecto es: **Arabica** con una probabilidad del 98.01%.

Consulta: Consumo de Robusta en Vietnam en el año 2018.
Resultado: El consumo histórico de café Robusta en 2018 fue de **59,505,240.00** unidades.
(Promedio anual si aplica: 3,967,016.00)

Consulta: Consumo de café en Estados Unidos en 2015.
Resultado: El consumo histórico de café 2015 fue de **2,840,952,000.00** unidades.
(Promedio anual si aplica: 51,653,672.73)

Consulta: Predice el consumo para 2030.
Res

Se importan librerias, luego se define una función que obtiene los datos de la query obteniendo intent, country, coffee_type y year, si la intención coincide con clasificar tipo de café, entonces se utiliza el modelo de clasificación preprocesando la entrada y luego realizando la predicción, si se detecta que se busca predecir consumo entonces se utiliza el modelo de predicción, si se detecta que se ingresó tipo de cafe, entonces se realiza la predicción del consumo según año, país y tipo de cafe, si no se detecta tipo, se realiza una predicción general para el páis y el año, si se detecta que es histórica se realiza el filtrado de información del dataset para obtener datos históricos, finalmente se realiza la prueba con multiples queries, algunas fallan para demostrar las limitaciones del sistema y las demás para demostrar su efectividad.

In [24]:
import joblib
import numpy as np

print("--- Exportando Modelos y Preprocesadores ---\n")

# 1. Exportar el Modelo de Clasificación (Keras)
model_cls.save('classification_model.h5')
print("Modelo de clasificación 'classification_model.h5' exportado exitosamente.")

# 2. Exportar el preprocesador de clasificación (ColumnTransformer)
joblib.dump(preprocessor_cls, 'preprocessor_cls.pkl')
print("Preprocesador de clasificación 'preprocessor_cls.pkl' exportado exitosamente.")

# 3. Exportar el LabelEncoder y las clases de tipo de café
joblib.dump(label_encoder, 'label_encoder.pkl')
np.save('coffee_type_classes.npy', coffee_type_classes)
print("LabelEncoder 'label_encoder.pkl' y clases 'coffee_type_classes.npy' exportados exitosamente.")

# 4. Exportar el Modelo de Regresión (Keras)
model_reg.save('regression_model.h5')
print("Modelo de regresión 'regression_model.h5' exportado exitosamente.")

# 5. Exportar el preprocesador de regresión (ColumnTransformer)
joblib.dump(preprocessor_reg, 'preprocessor_reg.pkl')
print("Preprocesador de regresión 'preprocessor_reg.pkl' exportado exitosamente.")

# 6. Exportar el escalador de la variable objetivo de regresión (StandardScaler)
joblib.dump(scaler_y_reg, 'scaler_y_reg.pkl')
print("Escalador de regresión 'scaler_y_reg.pkl' exportado exitosamente.")

# 7. Exportar el Tokenizador y Modelo BERT (Hugging Face)
tokenizer.save_pretrained('./bert_tokenizer')
model_bert.save_pretrained('./bert_model')
print("Tokenizador y Modelo BERT exportados exitosamente en sus respectivos directorios.")

print("\nTodos los componentes necesarios han sido exportados. Puedes encontrar los archivos en el entorno de Colab y descargarlos para tu aplicación Streamlit.")

--- Exportando Modelos y Preprocesadores ---

Modelo de clasificación 'classification_model.h5' exportado exitosamente.
Preprocesador de clasificación 'preprocessor_cls.pkl' exportado exitosamente.
LabelEncoder 'label_encoder.pkl' y clases 'coffee_type_classes.npy' exportados exitosamente.
Modelo de regresión 'regression_model.h5' exportado exitosamente.
Preprocesador de regresión 'preprocessor_reg.pkl' exportado exitosamente.
Escalador de regresión 'scaler_y_reg.pkl' exportado exitosamente.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tokenizador y Modelo BERT exportados exitosamente en sus respectivos directorios.

Todos los componentes necesarios han sido exportados. Puedes encontrar los archivos en el entorno de Colab y descargarlos para tu aplicación Streamlit.


Finalmente simplemente se exportan los scalers, modelos, preprocesadores etc... para su uso en streamlit